# تحلیل درخواست هزینه

این نوت‌بوک نشان می‌دهد چگونه عامل‌هایی ایجاد کنیم که از پلاگین‌ها برای پردازش هزینه‌های سفر از تصاویر رسیدهای محلی استفاده کرده، ایمیل درخواست هزینه تولید کنند و داده‌های هزینه را با استفاده از نمودار دایره‌ای مصورسازی کنند. عامل‌ها به صورت پویا توابع را بر اساس زمینه وظیفه انتخاب می‌کنند.

مراحل:
1. عامل OCR تصویر رسید محلی را پردازش کرده و داده‌های هزینه سفر را استخراج می‌کند.
2. عامل ایمیل یک ایمیل درخواست هزینه تولید می‌کند.

### نمونه‌ای از سناریوی هزینه سفر:
تصور کنید شما یک کارمند هستید که برای یک جلسه کاری در شهری دیگر سفر می‌کنید. شرکت شما سیاستی دارد که تمام هزینه‌های معقول مرتبط با سفر را بازپرداخت می‌کند. در اینجا تجزیه و تحلیل هزینه‌های احتمالی سفر آمده است:
- حمل و نقل:
بهای بلیط هواپیما برای رفت و برگشت از شهر محل زندگی به شهر مقصد.
تاکسی یا خدمات سفر اشتراکی به و از فرودگاه.
حمل و نقل محلی در شهر مقصد (مانند حمل و نقل عمومی، اجاره خودرو یا تاکسی).

- اقامت:
اقامت در هتل برای سه شب در یک هتل تجاری میان‌رده نزدیک محل جلسه.

- وعده‌های غذایی:
سهمیه روزانه غذا برای صبحانه، ناهار و شام، بر اساس سیاست شرکت برای هزینه روزانه.

- هزینه‌های متفرقه:
هزینه پارکینگ در فرودگاه.
هزینه‌های دسترسی اینترنت در هتل.
انعام یا هزینه خدمات کوچک.

- مستندسازی:
شما همه رسیدها (پروازها، تاکسی‌ها، هتل، وعده‌های غذایی و غیره) و گزارش هزینه تکمیل شده برای بازپرداخت ارسال می‌کنید.


## وارد کردن کتابخانه‌های مورد نیاز

وارد کردن کتابخانه‌ها و ماژول‌های لازم برای دفترچه یادداشت.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## تعریف مدل‌های هزینه

 یک مدل پایدنتیک برای هزینه‌های فردی و یک کلاس ExpenseFormatter برای تبدیل پرس‌وجوی کاربر به داده‌های ساختاریافته هزینه ایجاد کنید.

 هر هزینه به صورت زیر نمایش داده می‌شود:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## تعریف ابزارها - تولید ایمیل

یک تابع ابزار برای تولید ایمیلی جهت ارسال درخواست هزینه ایجاد کنید.
- این ابزار از دکوراتور `@tool` از چارچوب Microsoft Agent استفاده می‌کند.
- مبلغ کل هزینه‌ها را محاسبه کرده و جزئیات را در قالب متن ایمیل فرمت‌بندی می‌کند.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# ابزار استخراج هزینه‌های سفر از تصاویر رسید

ایجاد یک تابع ابزار برای استخراج هزینه‌های سفر از تصاویر رسید.
- این ابزار از `@tool` دکوراتور فریم‌ورک عامل مایکروسافت استفاده می‌کند.
- تصویر رسید را می‌خواند، آن را به base64 کدگذاری می‌کند و داده URI را برای تحلیل به عامل باز می‌گرداند.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## پردازش هزینه‌ها

عوامل را تعریف کنید و آن‌ها را با استفاده از `WorkflowBuilder` به یک جریان کاری متوالی وصل کنید.
- عامل OCR داده‌های ساختاری هزینه را از تصویر رسید با استفاده از ابزار `load_receipt_image` استخراج می‌کند.
- عامل ایمیل داده‌های استخراج شده را دریافت کرده و یک ایمیل حرفه‌ای درخواست هزینه با استفاده از ابزار `generate_expense_email` ایجاد می‌کند.
- `WorkflowBuilder` با `add_edge` یک خط لوله متوالی ایجاد می‌کند: عامل OCR → عامل ایمیل.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## تابع اصلی

جریان کاری متوالی را بسازید و آن را اجرا کنید تا تصویر رسید را پردازش کرده و ایمیل درخواست هزینه را ایجاد کند.


> **توجه:** این جریان کاری در حال حاضر تصویر رسید را به صورت متن base64 ارسال می‌کند، که اکثر مدل‌های چت (از جمله gpt-5-mini) آن را به عنوان تصویر در نظر نمی‌گیرند.
> همچنین ممکن است پنجره زمینه مدل را نیز پر کند. بهتر است از OCR با Azure AI Vision (یا ابزار OCR دیگر) استفاده کنید و فقط متن استخراج‌شده را ارسال کنید، یا کد را بازسازی کنید تا تصویر به صورت پیام `image_url` ارسال شود.
> اگر فقط می‌خواهید از بروز خطاهای زمینه‌ای جلوگیری کنید، تصویر رسید کوچکتری استفاده کنید یا مدلی با پنجره زمینه بزرگ‌تر را امتحان کنید.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
